# E791 $D^+\to\pi^-\pi^+\pi^+$ — standard fit closure with Square Dalitz normalization

This notebook repeats notebook 02 as an ordinary (non-CP) coefficient-fit closure test, changing only the normalization integration from `DalitzGrid` to `SquareDalitzGrid`.

All floating Cartesian coefficients are unbounded. To avoid the scale-runaway direction that appears when all non-reference coefficients become huge compared with the fixed $\rho(770)$ coefficient, the single randomized start is generated around the injected truth rather than over a very broad absolute box. This remains a closure test: every floating parameter is displaced from truth before minimization.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzGrid, DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, SquareDalitzGrid, enable_x64,
    invariants_to_square_dalitz, weighted_resample,
)

enable_x64()

## 1. Same E791 Fit-2 model as notebook 02

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.00, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR": phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}
truth = {}

def free_coefficient(name):
    x_truth, y_truth = truth_xy[name]
    truth[f"{name}.x"] = float(x_truth)
    truth[f"{name}.y"] = float(y_truth)
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, bounds=None, step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, bounds=None, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma", (0,1), coefficients["sigma"], mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0,1), coefficients["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0,1), coefficients["f0_980"], mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0,1), coefficients["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0,1), coefficients["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0,1), coefficients["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]
model = DecayModel(channel, components)
assert all(p.bounds is None for p in model.parameters if not p.fixed)

## 2. Uniform Square-Dalitz normalization grid

The integration grid is explicitly a uniform midpoint grid in $(m',\theta')\in[0,1]^2$. The non-uniform physical measure is carried only by the Jacobian stored in the sample weights.

In [ ]:
SQDP_N = 1000
SQDP_PAIR = (0, 1)
norm_sqdp = SquareDalitzGrid(
    channel.parent_mass, channel.daughter_masses,
    resolution=SQDP_N, pair=SQDP_PAIR, quadrature="midpoint",
).sample()
mp, tp = invariants_to_square_dalitz(
    norm_sqdp.s12, norm_sqdp.s13, norm_sqdp.s23,
    mother_mass=channel.parent_mass, masses=channel.daughter_masses, pair=SQDP_PAIR,
)
print(f"SqDP grid: {SQDP_N} x {SQDP_N} = {norm_sqdp.size:,} points")
print("mprime range:", float(mp.min()), float(mp.max()))
print("thetaprime range:", float(tp.min()), float(tp.max()))
print("Jacobian min/max:", float(norm_sqdp.weights.min()), float(norm_sqdp.weights.max()))

## 3. Visual check of the SqDP grid and Jacobian

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
stride = max(1, norm_sqdp.size // 40000)
axes[0].scatter(np.asarray(mp)[::stride], np.asarray(tp)[::stride], s=1)
axes[0].set(xlabel=r"$m'$", ylabel=r"$\theta'$", title="Uniform SqDP midpoint grid", xlim=(0,1), ylim=(0,1))
h = axes[1].hist2d(np.asarray(mp), np.asarray(tp), bins=100, range=((0,1),(0,1)), weights=np.asarray(norm_sqdp.weights))
fig.colorbar(h[3], ax=axes[1], label="summed Jacobian weight")
axes[1].set(xlabel=r"$m'$", ylabel=r"$\theta'$", title="Physical measure (Jacobian)", xlim=(0,1), ylim=(0,1))
plt.show()

## 4. Integration diagnostic: SqDP versus ordinary Dalitz grid

In [ ]:
GRID_N = 1000
norm_dp = DalitzGrid(channel.parent_mass, channel.daughter_masses, resolution=GRID_N).sample()
I_sqdp = float(model.prepare_cache(norm_sqdp, norm_sqdp).normalization(truth))
I_dp = float(model.prepare_cache(norm_dp, norm_dp).normalization(truth))
print("I(SqDP) =", I_sqdp)
print("I(DP)   =", I_dp)
print("SqDP/DP =", I_sqdp/I_dp)
print("relative difference =", (I_sqdp-I_dp)/I_dp)

## 5. Generate pseudo-data

In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000
pool = model.generate_phase_space(N_POOL, seed=2000)
truth_cache_pool = model.prepare_cache(pool, norm_sqdp)
truth_intensity, truth_normalization = truth_cache_pool.evaluate(truth)
target_weights = pool.weights * truth_intensity
data = weighted_resample(jax.random.key(791), pool, target_weights, N_DATA, replace=True)
print("truth normalization (SqDP) =", float(truth_normalization))

## 6. Ordinary likelihood and randomized start around truth

The parameters remain unbounded. The randomized start is deliberately displaced from truth but avoids initializing the minimizer in the asymptotic scale direction where all non-reference coefficients can become enormous while leaving the normalized PDF nearly unchanged.

In [ ]:
cache = model.prepare_cache(data, norm_sqdp)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

minimizer = Minimizer(nll, model.parameters, tolerance=0.1, verbose=2)

START_SEED = 314159
START_SIGMA = 0.35
rng = np.random.default_rng(START_SEED)
start_values = {
    p.name: float(truth[p.name] + rng.normal(0.0, START_SIGMA))
    for p in model.parameters if not p.fixed
}

print("randomized start sigma =", START_SIGMA)
print("NLL(truth) =", float(nll(truth)))
print("NLL(start) =", float(nll(start_values)))
gradient_check = minimizer.check_gradient(start_values, step_scale=1e-5, print_table=True)

## 7. Perform exactly one fit and reject runaway solutions

In [ ]:
result = minimizer.fit(start_values=start_values, simplex=False, ncall=100000)
fit_values = {p.name: float(result.values[p.name]) for p in model.parameters if not p.fixed}

nll_truth = float(nll(truth))
nll_start = float(nll(start_values))
nll_fit = float(result.fval)
max_abs_fit = max(abs(v) for v in fit_values.values())
runaway = (not np.isfinite(nll_fit)) or max_abs_fit > 100.0 or nll_fit > nll_start

print("valid          =", bool(result.valid))
print("NLL(start)     =", nll_start)
print("NLL(truth)     =", nll_truth)
print("NLL(fit)       =", nll_fit)
print("fit-truth NLL  =", nll_fit - nll_truth)
print("EDM            =", float(result.fmin.edm))
print("function calls =", int(result.nfcn))
print("max |fit par|  =", max_abs_fit)
print("runaway        =", runaway)

if runaway:
    raise RuntimeError(
        "Fit entered a runaway/non-physical scale direction; do not interpret it as closure."
    )

## 8. Closure table and pulls

In [ ]:
print(f"{'parameter':16s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
rows=[]
for p in model.parameters:
    if p.fixed: continue
    t=float(truth[p.name]); s=float(start_values[p.name]); f=float(result.values[p.name]); e=float(result.errors[p.name])
    pull=(f-t)/e
    rows.append((p.name,t,s,f,e,pull))
    print(f"{p.name:16s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")

pulls=np.asarray([r[5] for r in rows])
print("max |pull| =", float(np.max(np.abs(pulls))))
print("RMS pull   =", float(np.sqrt(np.mean(pulls**2))))

names=[r[0] for r in rows]
fig,ax=plt.subplots(figsize=(10,5))
ax.axhline(0); ax.axhline(1,ls='--'); ax.axhline(-1,ls='--')
ax.scatter(np.arange(len(names)),pulls)
ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names,rotation=60,ha='right')
ax.set_ylabel('(fit - truth) / error'); ax.set_title('SqDP-normalized coefficient closure')
plt.tight_layout(); plt.show()

## 9. Projection before and after the fit

In [ ]:
projection_cache = model.prepare_cache(pool, norm_sqdp)
def projection(values,bins):
    intensity,_=projection_cache.evaluate(values)
    w=np.asarray(pool.weights*intensity)
    h12,_=np.histogram(np.asarray(pool.s12),bins=bins,weights=w)
    h13,_=np.histogram(np.asarray(pool.s13),bins=bins,weights=w)
    return h12+h13
sdata=np.concatenate([np.asarray(data.s12),np.asarray(data.s13)])
bins=np.linspace(sdata.min(),sdata.max(),110); centers=0.5*(bins[:-1]+bins[1:])
hd,_=np.histogram(sdata,bins=bins); hs=projection(start_values,bins); hf=projection(fit_values,bins); ht=projection(truth,bins)
for h in (hs,hf,ht): h*=hd.sum()/h.sum()
fig,ax=plt.subplots(figsize=(10,5.5))
ax.errorbar(centers,hd,yerr=np.sqrt(np.maximum(hd,1)),fmt='.',label='toy')
ax.step(centers,hs,where='mid',label='start'); ax.step(centers,hf,where='mid',label='fit'); ax.step(centers,ht,where='mid',ls='--',label='truth')
ax.legend(); plt.show()

## Interpretation

The fit is considered a meaningful SqDP closure test only if it stays away from the asymptotic coefficient-scale direction, improves the randomized starting NLL, and returns finite uncertainties and reasonable pulls. If this stabilized single-start fit still fails, the next diagnostic should compare the normalization matrices element by element between SqDP and ordinary Dalitz integration.